# Laboratorio 2: 01 Descarga y Preprocesamiento

Este notebook implementa la **Parte 1** del laboratorio: la descarga de imágenes Sentinel-2 y el recorte al área de estudio (San Bernardo) para un análisis multitemporal (2016-2026).

## 1. Contexto de Adquisición

Se han seleccionado 6 periodos temporales para analizar la expansión urbana de San Bernardo:
- **2016**: Inicio de la serie temporal.
- **2018**: Seguimiento bienal.
- **2020**: Seguimiento bienal.
- **2022**: Seguimiento bienal.
- **2024**: Seguimiento bienal.
- **2026**: Estado actual.

**Método de descarga:** Copernicus Data Space Browser (Manual) y búsqueda vía STAC API (Programática).

In [3]:
import pystac_client
import pandas as pd
from pathlib import Path

# Configuración del Catálogo STAC
STAC_URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(STAC_URL)

# Área de Interés (AOI) - San Bernardo, Chile
bbox = [-70.85, -33.67, -70.60, -33.50]

years = [2016, 2018, 2020, 2022, 2024, 2026]
results = []

print("🔍 Buscando mejores productos Sentinel-2 L2A...")

for year in years:
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{year}-01-01/{year}-03-31",
        query={"eo:cloud_cover": {"lt": 5}},
        sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}]
    )
    
    items = list(search.items())
    if items:
        best = items[0]
        results.append({
            'Año': year,
            'Product ID': best.id,
            'Fecha': best.properties['datetime'],
            'Nubosidad (%)': best.properties['eo:cloud_cover'],
            'Sensor': best.properties['platform']
        })
    else:
        print(f"⚠️ No se encontraron imágenes para el año {year}")

df_metadata = pd.DataFrame(results)
df_metadata

🔍 Buscando mejores productos Sentinel-2 L2A...
⚠️ No se encontraron imágenes para el año 2016


,Año,Product ID,Fecha,Nubosidad (%),Sensor
0,2018,S2A_19HCD_20180213_2_L2A,2018-02-13T14:52:00.942000Z,0.003749,sentinel-2a
1,2020,S2A_19HCD_20200324_1_L2A,2020-03-24T14:52:00.842000Z,0.001125,sentinel-2a
2,2022,S2A_19HCD_20220314_0_L2A,2022-03-14T14:52:08.584000Z,0.001374,sentinel-2a
3,2024,S2B_19HCC_20240207_0_L2A,2024-02-07T14:52:18.714000Z,0.000803,sentinel-2b
4,2026,S2B_19HCC_20260117_0_L2A,2026-01-17T14:52:16.112000Z,0.025972,sentinel-2b


## 2. Inspección de Archivos Locales (data/raw)

Una vez descargados los productos `.SAFE` o las bandas individuales desde el Copernicus Browser, validamos su presencia en el directorio `data/raw`.

In [4]:
raw_path = Path('../data/raw')
files = list(raw_path.glob('*.tif')) + list(raw_path.glob('*.zip'))

if files:
    print(f"✅ Se han encontrado {len(files)} productos en {raw_path.absolute()}")
    for f in files:
        size_gb = f.stat().st_size / (1024**3)
        print(f"- {f.name} ({size_gb:.2f} GB)")
else:
    print("❌ No se han encontrado archivos en data/raw. Por favor, mueve tus descargas aquí.")

✅ Se han encontrado 6 productos en /home/jovyan/work/../data/raw
- S2B_MSIL2A_20210324T143729_N0500_R096_T19HCC_20230606T044004.SAFE.zip (1.13 GB)
- S2B_MSIL2A_20190514T143759_N0500_R096_T19HCC_20221205T004206.SAFE.zip (1.12 GB)
- S2B_MSIL2A_20171110T143729_N0500_R096_T19HCC_20231012T204000.SAFE.zip (1.16 GB)
- S2A_MSIL2A_20231015T143721_N0510_R096_T19HCC_20241108T065742.SAFE.zip (1.17 GB)
- S2C_MSIL2A_20260122T143741_N0511_R096_T19HCC_20260122T181009.zip (1.14 GB)
- S2A_MSIL2A_20160305T143722_N0500_R096_T19HCC_20231018T122504.SAFE.zip (1.14 GB)


## 3. Justificación de la Selección

- **Calidad Espectral**: Se seleccionaron productos **L2A (Bottom of Atmosphere)** para asegurar que los cambios detectados sean reales y no debidos a variaciones atmosféricas.
- **Fenología de Verano**: Enero y Marzo permiten capturar el suelo en su estado más seco, lo cual maximiza el contraste del **NDBI** para detectar concreto y asfalto frente a áreas agrícolas senescentes.
- **Consistencia del Sensor**: Se utilizaron Sentinel-2A y 2B de forma intercambiable gracias a la armonización de sus instrumentos MSI.